# Trophic model for gut data processing
This file is used to pre-process all data (especially Chia network and Thai Children data) into the format which is convenient for simulations.

In [194]:
########### Self-customized setting
import pandas as pd
import numpy as np

In [195]:
########### Generate (containing information of metabolite consumption and production)
k = 5 # Assumed number of cell types for current iteration of the model
net = pd.read_csv('../input-data/prior-recon-network.csv')
mean_net = net.groupby('celltypes_ID').mean()
valid_net = net[net.iloc[:, 3] != 0] # No transport reactions found
# selfish_net = mean_net[mean_net.iloc[:,1] == 2]
# i_selfish = selfish_net.index.values   #### i_selfish returns IDs of cell types don't generate byproducts
print(net.head())
print('###################################################################################################')

   celltypes_ID     Metabolite metabolites_ID  \
0             1        Glucose         C00031   
1             1  Ascorbic acid         C00072   
2             1            Glu         C00025   
3             1            Ala         C00041   
4             1      Succinate         C00042   

   edge_types (2 represents intake, 3 represents secretion and 5 represents intake and secretion)  \
0                                                  5                                                
1                                                  5                                                
2                                                  5                                                
3                                                  5                                                
4                                                  2                                                

                      Justification based on Recon3D  
0  Both uptake and secretion reactions foun

/var/folders/fb/tmrn3vrn74l8_gw5pjnml6vr0000gn/T/ipykernel_31203/1347496656.py:4: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  mean_net = net.groupby('celltypes_ID').mean()


In [209]:
########### Load names of all nodes in the prior network
names = pd.read_csv('../input-data/names_ID.txt',sep=':')
names.set_index('IDs', inplace=True)
print(names.head())
print('###################################################################################################')

########### Load names of all nodes in the prior network
i_intake = pd.read_csv('../input-data/nutrient_intake_ID.txt',sep=':')
i_intake = i_intake['IDs'].values
print(i_intake)
print('###################################################################################################')

########### Load mean cell abundance priors-randomly sampled from a uniform distribution ranged [0, 1)
celltype_all = pd.read_csv('../input-data/prior-celltype-abundance.txt', sep=',')
celltype_all.head()
### Randomly sampled relative abundances of the cell types
rand = np.random.uniform(0, 1, k)
celltype_all['Mean'] = rand/rand.sum()
# celltype_all = celltype_all.groupby('celltype_id').sum().iloc[1:,].reset_index()
celltype_ID = celltype_all['celltype_id']
#print((celltype_ID!=0).sum())
celltype = celltype_all[celltype_ID!=0].loc[:,'Mean']
celltype_ID = celltype_ID[celltype_ID!=0]
print(celltype.head())
print('###################################################################################################')

########### Load extracellular metabolome for all cell lines
ec_metabolome_all = pd.read_excel('../input-data/metabolome-immanuel.xlsx', sheet_name='ec_metabolites')
ec_metabolome_all = ec_metabolome_all.groupby('metabolites_ID').sum().iloc[1:,].reset_index()
ec_metabolome_ID = ec_metabolome_all['metabolites_ID']
#print((metabolome_ID!=0).sum())
ec_metabolome = ec_metabolome_all[ec_metabolome_all['U87MG']>=0] # Only take those metabolites for which initial extracellular measured concentration is non negative
ec_metabolome_ID = ec_metabolome['metabolites_ID']
# ec_metabolome_ID = ec_metabolome_ID[ec_metabolome_ID!=0]
print(ec_metabolome.head())
print('###################################################################################################')

########### Load intracellular metabolome for all cell lines
ic_metabolome_all = pd.read_excel('../input-data/metabolome-immanuel.xlsx', sheet_name='ic_metabolites')
ic_metabolome_all = ic_metabolome_all.iloc[1:, [0, 1, -2, -1]] # Selecting only the net gain column
ic_metabolome_all.columns = ['metabolite_name', 'metabolites_ID', 'U87MG', 'NSP']
ic_metabolome_ID = ic_metabolome_all['metabolites_ID']
ic_metabolome = ic_metabolome_all.iloc[np.isin(ic_metabolome_ID, ec_metabolome_ID), :].reset_index().drop(columns='index') # Only take those metabolites whose initial extracellular values are nonnegative
# ic_metabolome_ID = ic_metabolome_ID[np.isin(ic_metabolome_ID, ec_metabolome_ID)]
ic_metabolome_ID = ic_metabolome['metabolites_ID']
#print((metabolome_ID!=0).sum())
ic_metabolome = ic_metabolome.iloc[:,2:]
# ic_metabolome_ID = ic_metabolome['metabolites_ID']
print(ic_metabolome.head())

# intersected_names = np.intersect1d(celltype.columns.values, metabolome.columns.values)
# celltype = celltype[intersected_names]
# metabolome = metabolome[intersected_names]
# print('Intersection between celltype and metabolome:')
# print(celltype.head())
# print(metabolome.head())

            Names
IDs              
1     Cell type 1
2     Cell type 2
3     Cell type 3
4     Cell type 4
5     Cell type 5
###################################################################################################
['C00031' 'C00072' 'C00025' 'C00041' 'C00042' 'C00186' 'C00661' 'C00047'
 'C00183' 'C00026' 'C00049' 'C00123' 'C00037' 'C01384' 'C02170' 'C00022'
 'C00407' 'C00082' 'C00148' 'C00188' 'C00327' 'C00079' 'C00065' 'C00711'
 'C00073' 'C00064' 'C00135' 'C00062' 'C00078' 'C00158' 'C00383' 'C00152'
 'C00092' 'C00103' 'C00491']
###################################################################################################
0    0.270739
1    0.032036
2    0.165596
3    0.296236
4    0.235393
Name: Mean, dtype: float64
###################################################################################################
  metabolites_ID          Sum       U87MG         NSP
0         C00025   587.978577  302.213861  285.764716
1         C00026     6.686961    3.836234    2.8

/var/folders/fb/tmrn3vrn74l8_gw5pjnml6vr0000gn/T/ipykernel_31203/2487016029.py:29: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  ec_metabolome_all = ec_metabolome_all.groupby('metabolites_ID').sum().iloc[1:,].reset_index()


In [214]:
############ Filter out metabolites with negative initial values from net, i_intake and names
net = net[np.isin(net.loc[:, 'metabolites_ID'], ec_metabolome_ID)]
i_intake = i_intake[np.isin(i_intake, ec_metabolome_ID)]
names = names[np.isin(names.index, np.append(['1', '2', '3', '4', '5'], ec_metabolome_ID.to_numpy()))]


In [215]:
########### pickle all processed data which are useful for simulations
import pickle

pickle_out = open("cancer_network.pickle","wb")
#pickle.dump([net, i_selfish, i_intake, names], pickle_out)
pickle.dump([net, i_intake, names], pickle_out, protocol=2)
pickle_out.close()

pickle_out = open("data.pickle","wb")
#pickle.dump([metagenome_ID, metagenome, metabolome_ID, metabolome], pickle_out)
pickle.dump([celltype_ID, celltype, ec_metabolome_ID, ec_metabolome, ic_metabolome_ID, ic_metabolome], pickle_out, protocol=2)
pickle_out.close()